# Análisis de datos con `pandas`

**Notebook de práctica — Curso de Estadística Aplicada en Python**  
**Autor:** Roger M. López Justiniano

Este notebook transforma la presentación original en una guía práctica de trabajo con `pandas`. El objetivo no es memorizar funciones, sino entender un flujo de trabajo reproducible para analizar datos tabulares.

El flujo de trabajo que se usará es:

1. Importar datos.
2. Inspeccionar su estructura.
3. Seleccionar variables.
4. Filtrar observaciones.
5. Crear nuevas variables.
6. Ordenar resultados.
7. Agrupar y resumir información.
8. Visualizar patrones básicos.
9. Formular respuestas a preguntas empíricas.

Se trabajará principalmente con un dataset del FMI para responder preguntas sobre crecimiento, inflación, deuda, déficit fiscal y PIB per cápita.


## 0. Preparación del entorno

Una librería es un conjunto de funciones, clases y objetos que extienden las capacidades básicas de Python. En análisis de datos se utilizan frecuentemente `pandas`, `numpy` y `matplotlib`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


In [ ]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})


## 1. ¿Qué es un `DataFrame`?

Un `DataFrame` es una estructura de datos tabular. Tiene filas y columnas, como una hoja de cálculo, pero con una diferencia importante: cada columna tiene un tipo de dato y puede ser manipulada mediante operaciones programáticas.

Conceptualmente, se puede pensar en un `DataFrame` como una colección de variables observadas para distintas unidades. Si tenemos una tabla económica, cada fila puede representar un país en un año y cada columna una variable: inflación, crecimiento, deuda, continente, etc.


In [ ]:
ejemplo = pd.DataFrame({
    "Pais": ["Bolivia", "Bolivia", "Peru", "Peru", "Chile", "Chile"],
    "Año": [2022, 2023, 2022, 2023, 2022, 2023],
    "Inflacion": [1.7, 2.1, 8.5, 3.2, 11.6, 3.9],
    "Crecimiento": [3.6, 3.1, 2.7, -0.6, 2.4, 0.2]
})

ejemplo


In [ ]:
ejemplo.columns


In [ ]:
ejemplo.shape


El resultado de `.shape` tiene la forma:

$$
(n, k)
$$

donde $n$ es el número de filas u observaciones y $k$ es el número de columnas o variables.


## 2. Importación de datos

En un proyecto bien organizado, los datos suelen estar en una carpeta como `data/raw`. El archivo esperado es `Data_FMI.csv`. Para que el notebook sea más robusto, se prueban varias rutas posibles.


In [ ]:
posibles_rutas = [
    Path("../../data/raw/Data_FMI.csv"),
    Path("../data/raw/Data_FMI.csv"),
    Path("data/raw/Data_FMI.csv"),
    Path("Data_FMI.csv")
]

ruta_csv = None
for ruta in posibles_rutas:
    if ruta.exists():
        ruta_csv = ruta
        break

ruta_csv


In [ ]:
if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró Data_FMI.csv. Revisar que el archivo exista en data/raw o ajustar la ruta."
    )

data = pd.read_csv(ruta_csv)


## 3. Primer contacto con el dataset

La primera tarea no es calcular promedios ni hacer gráficos. La primera tarea es inspeccionar la estructura de los datos.


In [ ]:
data.head()


In [ ]:
data.tail()


In [ ]:
data.shape


In [ ]:
data.columns


In [ ]:
data.dtypes


In [ ]:
data.info()


### Valores faltantes

Un valor faltante indica ausencia de información. En `pandas`, muchos valores faltantes aparecen como `NaN`.


In [ ]:
data.isna().sum()


In [ ]:
faltantes = (
    data
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)

faltantes.columns = ["variable", "porcentaje_faltante"]
faltantes.head(10)


## 4. Lectura de archivos Excel

Los archivos Excel también son frecuentes en análisis de datos. Para leerlos se usa `pd.read_excel()`. Normalmente se requiere tener instalada la librería `openpyxl`.


In [ ]:
posibles_rutas_excel = [
    Path("../../data/raw/Data_FMI.xlsx"),
    Path("../data/raw/Data_FMI.xlsx"),
    Path("data/raw/Data_FMI.xlsx"),
    Path("Data_FMI.xlsx")
]

ruta_excel = None
for ruta in posibles_rutas_excel:
    if ruta.exists():
        ruta_excel = ruta
        break

ruta_excel


In [ ]:
if ruta_excel is not None:
    data_excel = pd.read_excel(ruta_excel)
    display(data_excel.head())
else:
    print("No se encontró Data_FMI.xlsx. Se continuará trabajando con el archivo CSV.")


In [ ]:
if ruta_excel is not None:
    hojas = pd.ExcelFile(ruta_excel).sheet_names
    display(hojas)
else:
    print("No hay archivo Excel disponible para inspeccionar hojas.")


## 5. Seleccionar columnas

Seleccionar columnas equivale a elegir variables. En análisis estadístico, esto es importante porque rara vez usamos todas las variables al mismo tiempo.


In [ ]:
data["GDPpc_usd"].head()


In [ ]:
type(data["GDPpc_usd"])


In [ ]:
data[["Pais", "Año", "GDPpc_usd"]].head()


In [ ]:
type(data[["GDPpc_usd"]])


In [ ]:
macro = data[[
    "Pais",
    "Año",
    "Continent",
    "GDPpc_usd",
    "rgrowth",
    "Inflacion",
    "Deficit_Fiscal",
    "Deuda"
]]

macro.head()


### Selección por posición con `.iloc`

`.iloc` permite seleccionar filas y columnas por posición. La sintaxis general es:

$$
\texttt{df.iloc[filas, columnas]}
$$


In [ ]:
data.iloc[0:5, 0:4]


### Selección por nombres con `.loc`

`.loc` permite seleccionar filas y columnas usando nombres o condiciones.


In [ ]:
data.loc[0:5, ["Pais", "Año", "Inflacion"]]


In [ ]:
data_sin_codigo = data.drop(columns=["ISO_Code"])
data_sin_codigo.head()


## 6. Filtrar filas

Filtrar filas significa conservar observaciones que cumplen una condición. En términos lógicos, una condición aplicada a una columna produce una máscara booleana:

$$
m_i =
\begin{cases}
\text{True}, & \text{si la observación } i \text{ cumple la condición} \\
\text{False}, & \text{si la observación } i \text{ no cumple la condición}
\end{cases}
$$

Luego `pandas` conserva las filas donde la máscara es `True`.


In [ ]:
ejemplo


In [ ]:
ejemplo["Año"] == 2023


In [ ]:
ejemplo[ejemplo["Año"] == 2023]


In [ ]:
data_2023 = data[data["Año"] == 2023]
data_2023.head()


In [ ]:
data_bolivia = data[data["Pais"] == "Bolivia"]
data_bolivia.head()


In [ ]:
data_bolivia_2000 = data[
    (data["Pais"] == "Bolivia") &
    (data["Año"] >= 2000)
]

data_bolivia_2000.head()


In [ ]:
data.loc[
    (data["Pais"] == "Bolivia") & (data["Año"] >= 2000),
    ["Pais", "Año", "Inflacion", "rgrowth", "Deuda"]
].head()


In [ ]:
data.query('Pais == "Bolivia" and Año >= 2000').head()


### Operadores frecuentes para filtrar

| Objetivo | Expresión en pandas |
|---|---|
| Igual a un valor | `df["x"] == valor` |
| Distinto de un valor | `df["x"] != valor` |
| Mayor que | `df["x"] > valor` |
| Mayor o igual que | `df["x"] >= valor` |
| Menor que | `df["x"] < valor` |
| Menor o igual que | `df["x"] <= valor` |
| Pertenece a una lista | `df["x"].isin(lista)` |
| No pertenece a una lista | `~df["x"].isin(lista)` |
| Es faltante | `df["x"].isna()` |
| No es faltante | `df["x"].notna()` |
| Y lógico | `(cond1) & (cond2)` |
| O lógico | `(cond1) | (cond2)` |
| Negación | `~(cond)` |


In [ ]:
paises = ["Bolivia", "Peru", "Chile", "Paraguay"]

data_vecinos = data[data["Pais"].isin(paises)]
data_vecinos[["Pais", "Año", "Inflacion", "rgrowth"]].head()


In [ ]:
data_vecinos_2022_2023 = data.loc[
    data["Pais"].isin(["Bolivia", "Peru", "Chile", "Paraguay"]) &
    data["Año"].isin([2022, 2023]),
    ["Pais", "Año", "Inflacion", "rgrowth", "Deuda"]
]

data_vecinos_2022_2023


## 7. Crear un dataset de trabajo

Para varios ejemplos posteriores usaremos Bolivia entre 1981 y 2023.


In [ ]:
data_bol = (
    data
    .loc[
        (data["Pais"] == "Bolivia") &
        (data["Año"] >= 1981) &
        (data["Año"] <= 2023)
    ]
    .copy()
)

data_bol.head()


In [ ]:
data_bol.tail()


In [ ]:
data_bol.shape


In [ ]:
data_bol[["Año", "GDPpc_usd", "rgrowth", "Inflacion", "Deficit_Fiscal", "Deuda"]].describe()


## 8. Crear y modificar variables

Crear variables es una de las operaciones más importantes en análisis de datos. Una variable nueva puede ser una transformación de escala, una categoría construida a partir de umbrales, un indicador binario o una variable temporal.


In [ ]:
data_bol_tmp = data_bol.copy()
data_bol_tmp["GDPpc_milesUSD"] = data_bol_tmp["GDPpc_usd"] / 1000

data_bol_tmp[["Pais", "Año", "GDPpc_usd", "GDPpc_milesUSD"]].head()


In [ ]:
data_bol_tmp = (
    data_bol
    .assign(GDPpc_milesUSD=lambda df: df["GDPpc_usd"] / 1000)
)

data_bol_tmp[["Pais", "Año", "GDPpc_usd", "GDPpc_milesUSD"]].head()


In [ ]:
data_bol_tmp = (
    data_bol
    .assign(inflacion_alta=lambda df: df["Inflacion"] > 10)
)

data_bol_tmp[["Pais", "Año", "Inflacion", "inflacion_alta"]].head(10)


In [ ]:
data_bol_tmp["inflacion_alta"].value_counts()


In [ ]:
data_bol_tmp = (
    data_bol
    .assign(
        cuentas_en=lambda df: np.where(
            df["Deficit_Fiscal"] < 0,
            "deficit",
            "superavit"
        )
    )
)

data_bol_tmp[["Pais", "Año", "Deficit_Fiscal", "cuentas_en"]].head()


### Precaución conceptual

El nombre `Deficit_Fiscal` puede inducir a confusión dependiendo de cómo esté construida la variable. Antes de interpretar el signo, se debe verificar la documentación del dataset.

- Si valores negativos representan déficit, entonces la clasificación anterior es correcta.
- Si valores positivos representan déficit como porcentaje del PIB, entonces la clasificación debería invertirse.
- Nunca se debe interpretar una variable económica solo por el nombre de la columna.


In [ ]:
condiciones = [
    data_bol["Inflacion"] < 5,
    (data_bol["Inflacion"] >= 5) & (data_bol["Inflacion"] < 10),
    (data_bol["Inflacion"] >= 10) & (data_bol["Inflacion"] < 50),
    data_bol["Inflacion"] >= 50
]

categorias = ["Baja", "Media", "Alta", "Muy alta"]

data_bol_inflacion = (
    data_bol
    .assign(
        Inflacion_Categoria=np.select(
            condiciones,
            categorias,
            default="Sin categoria"
        )
    )
)

data_bol_inflacion[["Pais", "Año", "Inflacion", "Inflacion_Categoria"]].head(15)


In [ ]:
data_bol_inflacion["Inflacion_Categoria"].value_counts()


## 9. Encadenamiento de operaciones

El encadenamiento permite escribir un flujo de trabajo como una secuencia de pasos. El código se lee de arriba hacia abajo.


In [ ]:
tmp1 = data_bol.copy()
tmp2 = tmp1[tmp1["Año"] >= 2000]
tmp3 = tmp2[["Pais", "Año", "Inflacion"]]

tmp3.head()


In [ ]:
(
    data_bol
    .loc[data_bol["Año"] >= 2000, ["Pais", "Año", "Inflacion"]]
    .head()
)


In [ ]:
resultado = (
    data_bol
    .assign(
        GDPpc_milesUSD=lambda df: df["GDPpc_usd"] / 1000,
        inflacion_alta=lambda df: df["Inflacion"] > 10
    )
    .query("Año >= 2000")
    .loc[:, ["Pais", "Año", "GDPpc_milesUSD", "Inflacion", "inflacion_alta"]]
)

resultado.head()


In [ ]:
(
    data_bol
    .assign(inflacion_alta=lambda df: df["Inflacion"] > 10)
    .loc[lambda df: df["inflacion_alta"]]
    .loc[:, ["Pais", "Año", "Inflacion", "inflacion_alta"]]
    .head()
)


## 10. Ordenar datos

Ordenar datos permite identificar extremos, rankings y casos relevantes.


In [ ]:
(
    data_bol
    .loc[:, ["Pais", "Año", "Inflacion"]]
    .sort_values("Inflacion", ascending=False)
    .head(10)
)


In [ ]:
(
    data_bol
    .loc[:, ["Pais", "Año", "Inflacion"]]
    .sort_values("Inflacion", ascending=True)
    .head(10)
)


In [ ]:
(
    data
    .loc[:, ["Pais", "Año", "Inflacion"]]
    .sort_values(["Pais", "Año"])
    .head(10)
)


In [ ]:
(
    data
    .loc[:, ["Pais", "Año", "Inflacion"]]
    .sort_values(["Pais", "Inflacion"], ascending=[True, False])
    .head(10)
)


## 11. Agrupar y resumir datos

Una de las operaciones más importantes de `pandas` es `.groupby()`. La lógica se conoce como **split-apply-combine**:

1. **Split**: dividir el dataset en grupos.
2. **Apply**: aplicar una función a cada grupo.
3. **Combine**: combinar los resultados en una nueva tabla.

Formalmente, si $G_g$ representa el conjunto de observaciones del grupo $g$, el promedio por grupo de una variable $x$ es:

$$
\bar{x}_g =
\frac{1}{n_g}
\sum_{i \in G_g} x_i
$$

donde $n_g$ es el número de observaciones dentro del grupo $g$.


In [ ]:
data_vecinos = (
    data
    .loc[
        data["Pais"].isin(["Bolivia", "Peru", "Chile", "Paraguay"]) &
        data["Año"].isin([2022, 2023]),
        ["Pais", "Año", "Inflacion", "rgrowth", "Deuda"]
    ]
    .sort_values(["Pais", "Año"])
)

data_vecinos


In [ ]:
grupos = data_vecinos.groupby("Pais")
grupos


In [ ]:
grupos.groups


In [ ]:
(
    data_vecinos
    .groupby("Pais")["Inflacion"]
    .mean()
)


In [ ]:
(
    data_vecinos
    .groupby("Pais")["Inflacion"]
    .mean()
    .reset_index()
)


In [ ]:
resumen_vecinos = (
    data_vecinos
    .groupby("Pais")
    .agg(
        Obs=("Inflacion", "size"),
        Inflacion_promedio=("Inflacion", "mean"),
        Inflacion_maxima=("Inflacion", "max"),
        Crecimiento_promedio=("rgrowth", "mean"),
        Deuda_promedio=("Deuda", "mean")
    )
    .reset_index()
)

resumen_vecinos


In [ ]:
(
    resumen_vecinos
    .sort_values("Inflacion_promedio", ascending=False)
)


In [ ]:
(
    data
    .loc[data["Año"].isin([2022, 2023])]
    .groupby(["Continent", "Año"])
    .agg(
        Inflacion_promedio=("Inflacion", "mean"),
        Crecimiento_promedio=("rgrowth", "mean"),
        Deuda_promedio=("Deuda", "mean"),
        Obs=("Pais", "nunique")
    )
    .reset_index()
    .sort_values(["Año", "Inflacion_promedio"], ascending=[True, False])
)


## 12. Resúmenes estadísticos básicos

El promedio de una variable $x_1, x_2, \dots, x_n$ se define como:

$$
\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i
$$

La mediana es el valor central de una distribución ordenada. Es menos sensible a valores extremos que el promedio.

La desviación estándar muestral mide dispersión alrededor de la media:

$$
s =
\sqrt{
\frac{1}{n-1}
\sum_{i=1}^{n}
(x_i - \bar{x})^2
}
$$


In [ ]:
data_bol["Inflacion"].mean()


In [ ]:
data_bol["Inflacion"].median()


In [ ]:
pd.DataFrame({
    "estadistico": ["media", "mediana", "maximo", "minimo"],
    "inflacion": [
        data_bol["Inflacion"].mean(),
        data_bol["Inflacion"].median(),
        data_bol["Inflacion"].max(),
        data_bol["Inflacion"].min()
    ]
})


In [ ]:
data_bol["Inflacion"].std()


In [ ]:
(
    data_bol
    .loc[:, ["GDPpc_usd", "rgrowth", "Inflacion", "Deficit_Fiscal", "Deuda"]]
    .agg(["count", "mean", "median", "std", "min", "max"])
)


## 13. Visualización básica con `pandas` y `matplotlib`

Los gráficos no reemplazan los resúmenes numéricos. Los complementan. Permiten detectar tendencias, cambios bruscos, valores extremos y relaciones entre variables.


In [ ]:
fig, ax = plt.subplots()

ax.plot(data_bol["Año"], data_bol["Inflacion"], marker="o")
ax.set_title("Inflación en Bolivia")
ax.set_xlabel("Año")
ax.set_ylabel("Inflación (%)")

plt.show()


In [ ]:
fig, ax = plt.subplots()

ax.axhline(0, linewidth=1)
ax.plot(data_bol["Año"], data_bol["rgrowth"], marker="o")
ax.set_title("Crecimiento real del PIB en Bolivia")
ax.set_xlabel("Año")
ax.set_ylabel("Crecimiento real del PIB (%)")

plt.show()


In [ ]:
fig, ax = plt.subplots()

ax.scatter(data["Deuda"], data["Inflacion"], alpha=0.5)
ax.set_title("Relación entre deuda e inflación")
ax.set_xlabel("Deuda pública (% del PIB)")
ax.set_ylabel("Inflación (%)")

plt.show()


In [ ]:
inflacion_continente_2023 = (
    data
    .loc[data["Año"] == 2023]
    .groupby("Continent")
    .agg(Inflacion_promedio=("Inflacion", "mean"))
    .reset_index()
    .sort_values("Inflacion_promedio", ascending=False)
)

inflacion_continente_2023


In [ ]:
fig, ax = plt.subplots()

ax.bar(
    inflacion_continente_2023["Continent"],
    inflacion_continente_2023["Inflacion_promedio"]
)

ax.set_title("Inflación promedio por continente, 2023")
ax.set_xlabel("Continente")
ax.set_ylabel("Inflación promedio (%)")
ax.tick_params(axis="x", rotation=45)

plt.show()


## 14. Ejemplos integrados

En esta sección se combinan selección, filtro, transformación, agrupación y visualización.


### Ejemplo 1: crecimiento promedio mundial en 2022 y 2023

¿Cuál fue el crecimiento promedio del PIB real mundial en los años 2022 y 2023?


In [ ]:
(
    data
    .loc[data["Año"].isin([2022, 2023])]
    .agg(Crecimiento_promedio=("rgrowth", "mean"))
)


In [ ]:
crecimiento_mundial = (
    data
    .loc[data["Año"].isin([2022, 2023])]
    .groupby("Año")
    .agg(
        Crecimiento_promedio=("rgrowth", "mean"),
        Obs=("rgrowth", "count")
    )
    .reset_index()
)

crecimiento_mundial


In [ ]:
fig, ax = plt.subplots()

ax.bar(
    crecimiento_mundial["Año"].astype(str),
    crecimiento_mundial["Crecimiento_promedio"]
)

ax.set_title("Crecimiento promedio del PIB real mundial")
ax.set_xlabel("Año")
ax.set_ylabel("Crecimiento promedio (%)")

plt.show()


### Ejemplo 2: crecimiento promedio por continente en 2023


In [ ]:
crecimiento_continente_2023 = (
    data
    .loc[data["Año"] == 2023]
    .groupby("Continent")
    .agg(
        Crecimiento_promedio=("rgrowth", "mean"),
        Obs=("rgrowth", "count")
    )
    .reset_index()
    .sort_values("Crecimiento_promedio", ascending=False)
)

crecimiento_continente_2023


In [ ]:
fig, ax = plt.subplots()

ax.bar(
    crecimiento_continente_2023["Continent"],
    crecimiento_continente_2023["Crecimiento_promedio"]
)

ax.set_title("Crecimiento promedio por continente, 2023")
ax.set_xlabel("Continente")
ax.set_ylabel("Crecimiento promedio (%)")
ax.tick_params(axis="x", rotation=45)

plt.show()


### Ejemplo 3: déficit fiscal y deuda en América


In [ ]:
(
    data
    .loc[data["Continent"] == "Americas"]
    .agg(
        Deficit_promedio=("Deficit_Fiscal", "mean"),
        Deuda_promedio=("Deuda", "mean"),
        Obs=("Pais", "count")
    )
)


In [ ]:
america_fiscal = (
    data
    .loc[data["Continent"] == "Americas"]
    .groupby("Año")
    .agg(
        Deficit_promedio=("Deficit_Fiscal", "mean"),
        Deuda_promedio=("Deuda", "mean")
    )
    .reset_index()
)

america_fiscal.tail()


In [ ]:
fig, ax = plt.subplots()

ax.plot(america_fiscal["Año"], america_fiscal["Deuda_promedio"], marker="o")
ax.set_title("Deuda promedio en América")
ax.set_xlabel("Año")
ax.set_ylabel("Deuda promedio (% del PIB)")

plt.show()


### Ejemplo 4: países con inflación mayor a 15% en 2023


In [ ]:
inflacion_alta_2023 = (
    data
    .loc[
        (data["Año"] == 2023) &
        (data["Inflacion"] > 15),
        ["Pais", "Continent", "Año", "Inflacion"]
    ]
    .sort_values("Inflacion", ascending=False)
)

inflacion_alta_2023


In [ ]:
(
    inflacion_alta_2023
    .groupby("Continent")
    .agg(Paises=("Pais", "count"))
    .reset_index()
    .sort_values("Paises", ascending=False)
)


### Ejemplo 5: deuda e inflación por categorías de deuda

Este ejercicio describe diferencias promedio entre grupos, pero no demuestra causalidad. El análisis descriptivo ayuda a formular preguntas, no a resolver por sí solo problemas causales.


In [ ]:
condiciones = [
    data["Deuda"] < 40,
    (data["Deuda"] >= 40) & (data["Deuda"] < 80),
    data["Deuda"] >= 80
]

categorias = ["Baja", "Media", "Alta"]

data_deuda = (
    data
    .assign(
        Deuda_Categoria=np.select(
            condiciones,
            categorias,
            default="Sin categoria"
        )
    )
)

data_deuda[["Pais", "Año", "Deuda", "Deuda_Categoria", "Inflacion"]].head()


In [ ]:
inflacion_por_deuda = (
    data_deuda
    .loc[data_deuda["Deuda_Categoria"] != "Sin categoria"]
    .groupby("Deuda_Categoria")
    .agg(
        Inflacion_promedio=("Inflacion", "mean"),
        Inflacion_mediana=("Inflacion", "median"),
        Obs=("Inflacion", "count")
    )
    .reset_index()
)

inflacion_por_deuda


In [ ]:
fig, ax = plt.subplots()

ax.bar(
    inflacion_por_deuda["Deuda_Categoria"],
    inflacion_por_deuda["Inflacion_promedio"]
)

ax.set_title("Inflación promedio por categoría de deuda")
ax.set_xlabel("Categoría de deuda")
ax.set_ylabel("Inflación promedio (%)")

plt.show()


## 15. Ejercicios guiados

En esta sección se proponen ejercicios con una estructura sugerida. La recomendación es intentar resolverlos antes de ejecutar la solución.


### Ejercicio 1: observaciones del año 2023


In [ ]:
data_23 = data.loc[data["Año"] == 2023].copy()
data_23.head()


### Ejercicio 2: observaciones de Bolivia


In [ ]:
data_bolivia = data.loc[data["Pais"] == "Bolivia"].copy()
data_bolivia.head()


### Ejercicio 3: Bolivia desde el año 2000


In [ ]:
data_bol_2000 = data.loc[
    (data["Pais"] == "Bolivia") &
    (data["Año"] >= 2000)
].copy()

data_bol_2000.head()


### Ejercicio 4: años de Bolivia con inflación mayor a 10%


In [ ]:
(
    data
    .loc[
        (data["Pais"] == "Bolivia") &
        (data["Inflacion"] > 10),
        ["Pais", "Año", "Inflacion"]
    ]
    .sort_values("Inflacion", ascending=False)
)


### Ejercicio 5: PIB per cápita promedio de Bolivia entre 2010 y 2023


In [ ]:
(
    data
    .loc[
        (data["Pais"] == "Bolivia") &
        (data["Año"].between(2010, 2023)),
        ["GDPpc_usd"]
    ]
    .mean()
)


### Ejercicio 6: PIB per cápita promedio por década


In [ ]:
bolivia_decadas = (
    data
    .loc[data["Pais"] == "Bolivia"]
    .assign(Decada=lambda df: (df["Año"] // 10) * 10)
    .groupby("Decada")
    .agg(
        GDPpc_promedio=("GDPpc_usd", "mean"),
        Obs=("GDPpc_usd", "count")
    )
    .reset_index()
)

bolivia_decadas


### Ejercicio 7: 10 países con mayor PIB per cápita en 2023


In [ ]:
(
    data
    .loc[data["Año"] == 2023, ["Pais", "Continent", "Año", "GDPpc_usd"]]
    .dropna(subset=["GDPpc_usd"])
    .sort_values("GDPpc_usd", ascending=False)
    .head(10)
)


### Ejercicio 8: 10 países con menor PIB per cápita en 2023


In [ ]:
(
    data
    .loc[data["Año"] == 2023, ["Pais", "Continent", "Año", "GDPpc_usd"]]
    .dropna(subset=["GDPpc_usd"])
    .sort_values("GDPpc_usd", ascending=True)
    .head(10)
)


### Ejercicio 9: crecimiento, inflación y deuda por continente en 2023


In [ ]:
(
    data
    .loc[data["Año"] == 2023]
    .groupby("Continent")
    .agg(
        Crecimiento_promedio=("rgrowth", "mean"),
        Inflacion_promedio=("Inflacion", "mean"),
        Deuda_promedio=("Deuda", "mean"),
        Paises=("Pais", "nunique")
    )
    .reset_index()
    .sort_values("Crecimiento_promedio", ascending=False)
)


### Ejercicio 10: clasificar inflación en 2023


In [ ]:
data_2023_inflacion = (
    data
    .loc[data["Año"] == 2023]
    .assign(
        Inflacion_Categoria=lambda df: np.select(
            [
                df["Inflacion"] < 5,
                (df["Inflacion"] >= 5) & (df["Inflacion"] < 10),
                (df["Inflacion"] >= 10) & (df["Inflacion"] < 50),
                df["Inflacion"] >= 50
            ],
            ["Baja", "Media", "Alta", "Muy alta"],
            default="Sin categoria"
        )
    )
)

(
    data_2023_inflacion
    .groupby("Inflacion_Categoria")
    .agg(Paises=("Pais", "count"))
    .reset_index()
    .sort_values("Paises", ascending=False)
)


## 16. Ejercicios para resolver

Las siguientes celdas están vacías para que puedan resolverse durante la clase o como práctica individual.


### Ejercicio A. Inflación promedio por país para vecinos ampliados de Bolivia, 2010-2023


In [ ]:
# Escribir la solución aquí


### Ejercicio B. Cinco años de menor crecimiento económico de Bolivia


In [ ]:
# Escribir la solución aquí


### Ejercicio C. Deuda promedio por continente en 2023


In [ ]:
# Escribir la solución aquí


### Ejercicio D. Clasificar crecimiento real del PIB en 2023


In [ ]:
# Escribir la solución aquí


### Ejercicio E. Gráfico de dispersión: deuda y crecimiento en 2023


In [ ]:
# Escribir la solución aquí


## 17. Soluciones de los ejercicios finales


### Solución A


In [ ]:
paises_vecinos_ampliado = ["Bolivia", "Peru", "Chile", "Paraguay", "Argentina", "Brazil"]

(
    data
    .loc[
        data["Pais"].isin(paises_vecinos_ampliado) &
        data["Año"].between(2010, 2023)
    ]
    .groupby("Pais")
    .agg(
        Inflacion_promedio=("Inflacion", "mean"),
        Inflacion_mediana=("Inflacion", "median"),
        Obs=("Inflacion", "count")
    )
    .reset_index()
    .sort_values("Inflacion_promedio", ascending=False)
)


### Solución B


In [ ]:
(
    data
    .loc[
        data["Pais"] == "Bolivia",
        ["Pais", "Año", "rgrowth", "Inflacion", "Deuda"]
    ]
    .dropna(subset=["rgrowth"])
    .sort_values("rgrowth", ascending=True)
    .head(5)
)


### Solución C


In [ ]:
(
    data
    .loc[data["Año"] == 2023]
    .groupby("Continent")
    .agg(
        Deuda_promedio=("Deuda", "mean"),
        Deuda_mediana=("Deuda", "median"),
        Paises=("Pais", "nunique")
    )
    .reset_index()
    .sort_values("Deuda_promedio", ascending=False)
)


### Solución D


In [ ]:
data_crecimiento_2023 = (
    data
    .loc[data["Año"] == 2023]
    .assign(
        crecimiento_tipo=lambda df: np.select(
            [
                df["rgrowth"] < 0,
                (df["rgrowth"] >= 0) & (df["rgrowth"] < 2),
                (df["rgrowth"] >= 2) & (df["rgrowth"] < 5),
                df["rgrowth"] >= 5
            ],
            ["Recesion", "Bajo", "Moderado", "Alto"],
            default="Sin categoria"
        )
    )
)

(
    data_crecimiento_2023
    .groupby("crecimiento_tipo")
    .agg(Paises=("Pais", "count"))
    .reset_index()
    .sort_values("Paises", ascending=False)
)


### Solución E


In [ ]:
data_scatter_2023 = (
    data
    .loc[
        data["Año"] == 2023,
        ["Pais", "Deuda", "rgrowth"]
    ]
    .dropna()
)

fig, ax = plt.subplots()

ax.scatter(data_scatter_2023["Deuda"], data_scatter_2023["rgrowth"], alpha=0.6)
ax.axhline(0, linewidth=1)
ax.set_title("Deuda y crecimiento real del PIB, 2023")
ax.set_xlabel("Deuda pública (% del PIB)")
ax.set_ylabel("Crecimiento real del PIB (%)")

plt.show()


## 18. Buenas prácticas al trabajar con `pandas`

1. Inspeccionar antes de analizar: usar `.head()`, `.info()`, `.dtypes` e `.isna().sum()`.
2. Usar nombres claros: `data_bol`, `inflacion_continente_2023`, `crecimiento_mundial`.
3. Evitar modificar el dataset original sin necesidad: usar `.copy()` cuando se cree un subconjunto de trabajo.
4. Preferir flujos explícitos con `.loc` cuando se filtran filas y se seleccionan columnas.
5. Revisar resultados intermedios para evitar errores silenciosos.


In [ ]:
# Ejemplo de flujo explícito
(
    data
    .loc[data["Año"] == 2023, ["Pais", "Inflacion"]]
    .head()
)


## 19. Resumen

| Tarea | Código típico |
|---|---|
| Importar CSV | `pd.read_csv(ruta)` |
| Importar Excel | `pd.read_excel(ruta)` |
| Ver primeras filas | `df.head()` |
| Ver últimas filas | `df.tail()` |
| Ver dimensiones | `df.shape` |
| Ver tipos de datos | `df.dtypes` |
| Resumen general | `df.info()` |
| Seleccionar columnas | `df[["col1", "col2"]]` |
| Filtrar filas | `df.loc[condicion]` |
| Filtrar y seleccionar | `df.loc[condicion, columnas]` |
| Crear variables | `df.assign(nueva=lambda df: ...)` |
| Ordenar | `df.sort_values("col")` |
| Agrupar | `df.groupby("grupo")` |
| Resumir | `.agg(nombre=("variable", "funcion"))` |
| Reiniciar índice | `.reset_index()` |
| Valores faltantes | `df.isna().sum()` |
| Conteos | `df["col"].value_counts()` |

`pandas` no debe verse como una lista de comandos aislados. Es mejor entenderlo como una gramática práctica para trabajar con datos tabulares. La práctica constante permite internalizar un flujo de trabajo reproducible y eficiente.
